In [ ]:
from project_utils import load_table, input_path, output_path, calculateCI, mean_error, standard_error, percentile_column, positive_fraction, SEED
import numpy as np
np.random.seed(SEED)


In [ ]:
# author Alex J. Yang
import numpy as np
import pandas as pd
import itertools


In [ ]:
import scipy
import seaborn as sns
import matplotlib.pyplot as plt
import statsmodels.api as sm


In [ ]:
df = load_table('papers')


In [ ]:
df.dropna(subset = 'Disruption',inplace = True)


In [ ]:
# calculate CI using bootstrap

convert_column_to_percentile = percentile_column


In [ ]:
def yearGroup(x):
    if x<1920:
        return '1900-1919'
    elif x<1940:
        return '1920-1939'
    elif x<1960:
        return '1940-1959'
    elif x<1980:
        return '1960-1979'
    else:
        return '1980-1999'
    
def delayGroup(x):
    if x<2:
        return x
    elif x<10:
        return '2-9'
    elif x<20:
        return '10-19'
    elif x<50:
        return '20-49'
    else:
        return '50+'
df['yearGroup'] = df['Year'].apply(yearGroup)
df['delayGroup'] = df['SB_T'].apply(delayGroup)


In [ ]:
df1= df[(df['type_Concave'] == 0)&(df['type_Immediacy'] == 0)].copy()
df2 = df[df['type_Concave'] == 1].copy()
df3 = df[df['type_Immediacy'] == 1].copy()


In [ ]:
convert_column_to_percentile(df1, 'SB_B')
def BpGroup(x):
    if pd.isna(x):
        return np.nan
    if x<5:
        return 5
    elif x<50:
        return 50
    elif x<95:
        return 95
    else:
        return 100
df1['SB_B_p'] = df1['SB_B_percentile'].apply(BpGroup)


In [ ]:
df.columns


# Disruption


In [ ]:
dflst = [df , df1 , df2 , df3]


In [ ]:
x = range(6)
typelst = [0,1,'2-9', '10-19', '20-49','50+']
colum = 'Disruption'
y = np.array([df[df['delayGroup']==x][colum].mean() for x in typelst])
CI = np.array([mean_error(df[df['delayGroup']==x][colum].values) for x in typelst]).T


In [ ]:
f, ax = plt.subplots(figsize=(9, 7))
plt.bar(x , y , yerr = CI , capsize = 15 ,color='#424949', alpha = .9)
plt.yticks(fontsize=20,)
plt.xticks(x,typelst,fontsize=20,)
plt.ylim(0,0.3)
plt.ylabel('Average Disruption',fontsize = 20)
plt.xlabel(r'$𝑡_𝛼$',fontsize = 30)
for _ in range(len(y)):
    if y[_]>0.04:
        plt.text(x[_], y[_]-0.02 , str(y[_].round(3)),fontsize = 16,c = 'white', ha='center', fontweight='bold')
    else:
        plt.text(x[_], y[_]+0.005 , str(y[_].round(3)),fontsize = 16,c = 'black', ha='center')


In [ ]:
f, ax = plt.subplots(figsize=(7, 7))
plt.scatter(df['SB_T'], df['Disruption'], c = 'black', alpha = 0.01)
sns.regplot(data = df, y='Disruption' , x='SB_T',x_ci = 95 , n_boot = 100 ,seed = 0 , color = '#E74C3C',scatter = False)
plt.ylim(-1,1)
plt.xlim(0,100)
plt.yticks(fontsize=20,)
plt.xticks(fontsize=20,)
plt.ylabel('Disruption',fontsize = 30)
plt.xlabel(r'$𝑡_𝛼$',fontsize = 30)
plt.show()


In [ ]:
x = range(6)
typelst = ['convex','immediacy','0-5','5-50','50-95','95-100']
CClst = [5,50,95,100]
colum = 'Disruption'
y1 = np.array([df1[df1['SB_B_p']==x][colum].mean() for x in CClst])
CI1 = np.array([mean_error(df1[df1['SB_B_p']==x][colum].values) for x in CClst]).T

y2 =  np.array(df2[colum].mean())
y3 =  np.array(df3[colum].mean())
CI2 = np.array(mean_error(df2[colum].values))
CI3 = np.array(mean_error(df3[colum].values))

y =  np.append(np.array([y2,y3]),y1)
CI = np.column_stack((CI2, CI3, CI1))


In [ ]:
f, ax = plt.subplots(figsize=(10, 7))
plt.bar(x , y , yerr = CI , capsize = 15 ,color=['#117A65','#A93226']+['#2471A3']*4, alpha = .9)
plt.yticks(fontsize=20,)
plt.xticks(x,typelst,fontsize=15,)
plt.ylim(0,0.3)
plt.ylabel('Average Disruption',fontsize = 20)
plt.xlabel('B (percentile)',fontsize = 20)
for _ in range(len(y)):
    if y[_]>0.04:
        plt.text(x[_], y[_]-0.02 , str(y[_].round(3)),fontsize = 16,c = 'white', ha='center', fontweight='bold')
    else:
        plt.text(x[_], y[_]+0.005 , str(y[_].round(3)),fontsize = 16,c = 'black', ha='center')


In [ ]:
f, ax = plt.subplots(figsize=(7, 7))
plt.scatter(df1['SB_B_percentile'], df1['Disruption'], c = 'black', alpha = 0.01)
sns.regplot(data = df1, y='Disruption' , x='SB_B_percentile',x_ci = 95 , n_boot = 100 ,seed = 0 , color = '#E74C3C',scatter = False)
plt.ylim(-1,1)
plt.xlim(0,100)
plt.yticks(fontsize=20,)
plt.xticks(fontsize=20,)
plt.ylabel('Disruption',fontsize = 30)
plt.xlabel('B (percentile)',fontsize = 30)
plt.show()
